# Day 067 — Exercise 1: Image to Base64

**What you'll build:** `image_to_base64(img, format)` — encode a PIL Image as a base64 string ready for the Ollama vision API.

**Why it matters:** Every vision LLM API accepts images as base64 strings embedded in JSON. This function is the encoding bridge between Pillow and Ollama (and any other vision API). You will reuse it in every subsequent exercise today and in Days 69, 71, and 76.

In [ ]:
import io
import base64
from PIL import Image

_rgb  = Image.new('RGB',  (100, 80), color=(255, 0, 0))
_rgba = Image.new('RGBA', (50, 50),  color=(0, 200, 100, 128))


## Task

Implement `image_to_base64(img, format='PNG') -> str`:

1. Create `io.BytesIO()`
2. If `format` is `'JPEG'`/`'JPG'` and `img.mode` is `'RGBA'` or `'P'`, convert to `'RGB'` first
3. Call `out.save(buf, format=format)`
4. Return `base64.b64encode(buf.getvalue()).decode()`

## Your Implementation

In [ ]:
def image_to_base64(img: Image.Image, format: str = 'PNG') -> str:
    """Encode a PIL Image as a base64 string.

    For JPEG format, automatically converts RGBA and P mode images to RGB.

    Args:
        img:    PIL Image to encode
        format: image format ('PNG', 'JPEG', etc.)
    Returns:
        Base64-encoded string — no data URI prefix (just the raw base64 data)
    """
    raise NotImplementedError


In [ ]:
def image_to_base64(img: Image.Image, format: str = 'PNG') -> str:
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()


## Automated checks

In [ ]:
score, total = 0, 5
try:
    result = image_to_base64(_rgb)
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    score += 1; print("\u2705 returns a string")

    assert len(result) > 0, "base64 string should not be empty"
    score += 1; print("\u2705 non-empty base64 string")

    # Decode and verify it is a valid PNG
    raw = base64.b64decode(result)
    assert raw[:4] == b'\x89PNG', f"Not PNG: {raw[:4]!r}"
    score += 1; print("\u2705 decoded bytes are valid PNG")

    # JPEG encoding works
    jpeg_b64 = image_to_base64(_rgb, 'JPEG')
    assert isinstance(jpeg_b64, str) and len(jpeg_b64) > 0
    jpeg_raw = base64.b64decode(jpeg_b64)
    assert jpeg_raw[:2] == b'\xff\xd8', f"Not JPEG: {jpeg_raw[:2]!r}"
    score += 1; print("\u2705 JPEG encoding produces valid JPEG bytes")

    # RGBA → JPEG auto-converts without error
    rgba_b64 = image_to_base64(_rgba, 'JPEG')
    assert isinstance(rgba_b64, str) and len(rgba_b64) > 0
    score += 1; print("\u2705 RGBA image encodes to JPEG without error")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def image_to_base64(img: Image.Image, format: str = 'PNG') -> str:
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()
```

**Why `buf.getvalue()` not `buf.read()`?** `getvalue()` returns the entire buffer contents regardless of the current position — no `seek(0)` needed. `buf.read()` reads from the current position; after a write the position is at the end, so `read()` would return empty bytes without a preceding `seek(0)`. `getvalue()` is the correct choice when you only need the full byte string.

</details>